In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 Apply FIXED winning PARAMETER-FREE rule per system -- already selected from
 TRAIN -- to the 5-fold held-out TEST (20%) subsets.

 No sweep: each system uses exactly the single rule that won on TRAIN (there
 was never a p to sweep for this family in the first place). This is the
 honest generalization check -- the winners were chosen without ever looking
 at the test folds; this script is the first time the test folds are touched.

 Output columns: Fold | System | Rule | EER (%) | Rank-1 @ FAR=1% | N_test
================================================================================

No command-line arguments: every setting is a constant in the CONFIGURATION
section below, including BEST_CONFIGS -- the winning rule per system, taken
from the seed=62 TRAIN run (generate_paramfree_seed62_report.py).

USAGE
-----
    python evaluate_paramfree_best_configs_on_test.py
================================================================================
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Callable, Dict, List

import numpy as np
import pandas as pd
from scipy.io import loadmat
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

sys.path.insert(0, str(Path.cwd()))

from evaluate_pdep_train_5fold import (  # noqa: E402
    DataConfig,
    SplitConfig,
    SystemDefinition,
    SYSTEM_DEFINITIONS,
    compute_eer,
    compute_rank1_at_far,
    is_valid,
    make_folds,
    make_normalizer,
    split_genuine_impostor,
    submatrix,
    train_indices_for_fold,
)


# ==============================================================================
#  CONFIGURATION -- no argparse. Edit constants here and re-run.
# ==============================================================================

MAT_FILE = "870d37a7-673d-49ff-8a98-47b9b519237f.mat"
SEED = 62          # MUST match the seed used to derive BEST_CONFIGS below,
                   # so the same 5 folds are reproduced on the TEST side.
N_FOLDS = 5
FAR_TARGET = 0.01
OUTPUT_DIR = "."
OUTPUT_FILENAME_TEMPLATE = f"./Results/ParamFree_BestConfigs_TEST_seed{SEED}.xlsx"

# Winning rule per system, selected on TRAIN (seed=62) by
# generate_paramfree_seed62_report.py -- one fixed value, no p involved.
BEST_CONFIGS: Dict[str, str] = {
    "S1": "Rule3",
    "S2": "Rule5",
    "S3": "Rule5",
    "S4": "Rule5",
    "S5": "Rule5",
    "S6": "Rule8",
    "S7": "Rule5",
    "S8": "Rule5",
}


# ==============================================================================
#  PARAMETER-FREE FUSION RULES (same formulas as the TRAIN-side script)
# ==============================================================================

def rule1(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = xy * max(x,y)"""
    return x * y * np.maximum(x, y)


def rule2(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = xy * (x+y-xy)"""
    xy = x * y
    return xy * (x + y - xy)


def rule3(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = xy * min(x,y)"""
    return x * y * np.minimum(x, y)


def rule4(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = xy + (xy)^2 - xy*min(x,y)"""
    xy = x * y
    return xy + xy ** 2 - xy * np.minimum(x, y)


def rule5(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = (x+y-xy) + min(x,y) - (x+y-xy)*min(x,y)"""
    prob_sum = x + y - x * y
    mn = np.minimum(x, y)
    return prob_sum + mn - prob_sum * mn


def rule6(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = x+y - xy*(x+y-xy)"""
    xy = x * y
    return (x + y) - xy * (x + y - xy)


def rule7(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = (x+y-xy) + max(x,y) - (x+y-xy)*max(x,y)"""
    prob_sum = x + y - x * y
    mx = np.maximum(x, y)
    return prob_sum + mx - prob_sum * mx


def rule8(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = (x+y-xy) * max(x,y)"""
    prob_sum = x + y - x * y
    return prob_sum * np.maximum(x, y)


PARAMFREE_RULES: Dict[str, Callable[[np.ndarray, np.ndarray], np.ndarray]] = {
    "Rule1": rule1, "Rule2": rule2, "Rule3": rule3, "Rule4": rule4,
    "Rule5": rule5, "Rule6": rule6, "Rule7": rule7, "Rule8": rule8,
}


def apply_paramfree_rule(rule_name: str, x: np.ndarray, y: np.ndarray,
                         invalid_value: float) -> np.ndarray:
    """Fuse two normalized matrices with a parameter-free rule (no p)."""
    valid = is_valid(x, invalid_value) & is_valid(y, invalid_value)
    x_safe = np.where(valid, x, 0.0)
    y_safe = np.where(valid, y, 0.0)

    with np.errstate(invalid="ignore", divide="ignore"):
        fused = PARAMFREE_RULES[rule_name](x_safe, y_safe)

    fused = np.clip(fused, 0.0, 1.0)
    return np.where(valid, fused, invalid_value)


def build_system_matrix_paramfree(system: SystemDefinition,
                                  normalized: Dict[str, np.ndarray],
                                  rule_name: str,
                                  invalid_value: float) -> np.ndarray:
    """Fuse the system's two modalities; cascade a third stage if defined."""
    mod_a, mod_b = system.stage1
    fused = apply_paramfree_rule(rule_name, normalized[mod_a], normalized[mod_b],
                                 invalid_value)
    if system.is_cascaded:
        fused = apply_paramfree_rule(rule_name, fused,
                                     normalized[system.stage2_modality],
                                     invalid_value)
    return fused


# ==============================================================================
#  STEP 1 -- EVALUATE EACH FIXED WINNING RULE ON ITS FOLD'S HELD-OUT TEST SET
# ==============================================================================

def evaluate_best_configs_on_test() -> pd.DataFrame:
    """
    For every fold and every system, apply that system's single fixed
    winning rule -- unchanged across all folds -- to the fold's TEST
    submatrix. Normalization is fit on the fold's TRAIN submatrix only
    (same as during training) and applied to TEST: the rule choice never
    touches test data, and neither does the normalization scale.
    """
    data_cfg = DataConfig(mat_file=MAT_FILE)
    split_cfg = SplitConfig(n_folds=N_FOLDS, seed=SEED, shuffle=True)

    contents = loadmat(MAT_FILE)
    raw_matrices = {
        modality: np.asarray(contents[key], dtype=float)
        for modality, key in data_cfg.matrix_keys.items()
    }
    n_subjects = next(iter(raw_matrices.values())).shape[0]
    folds = make_folds(n_subjects, split_cfg)

    records: List[dict] = []
    for fold_idx in range(N_FOLDS):
        test_ids = folds[fold_idx]
        train_ids = train_indices_for_fold(folds, fold_idx, n_subjects)

        normalized_test: Dict[str, np.ndarray] = {}
        for modality, matrix in raw_matrices.items():
            train_block = submatrix(matrix, train_ids)
            test_block = submatrix(matrix, test_ids)
            normalizer = make_normalizer(data_cfg, label=modality)
            normalizer.fit(train_block)
            normalized_test[modality] = normalizer.transform(test_block)

        for system_name, rule_name in BEST_CONFIGS.items():
            system = SYSTEM_DEFINITIONS[system_name]
            fused_test = build_system_matrix_paramfree(
                system, normalized_test, rule_name, data_cfg.invalid_value
            )
            genuine, impostor = split_genuine_impostor(fused_test, data_cfg.invalid_value)

            records.append({
                "Fold": fold_idx + 1,
                "System": system_name,
                "Rule": rule_name,
                "EER (%)": round(compute_eer(genuine, impostor) * 100, 4),
                "Rank-1 @ FAR=1%": round(
                    compute_rank1_at_far(fused_test, FAR_TARGET, data_cfg.invalid_value) * 100,
                    4,
                ),
                "N_test": len(test_ids),
            })

        print(f"  Fold {fold_idx + 1}/{N_FOLDS} done ({len(test_ids)} test subjects).")

    report = pd.DataFrame.from_records(records)
    system_order = list(BEST_CONFIGS.keys())
    report["System"] = pd.Categorical(report["System"], system_order, ordered=True)
    return report.sort_values(["System", "Fold"]).reset_index(drop=True)


# ==============================================================================
#  STEP 2 -- WRITE THE FORMATTED, SEED-TAGGED XLSX
# ==============================================================================

def write_report(report: pd.DataFrame) -> Path:
    font_name = "Arial"
    theme_color = "6A1B9A"  # purple, distinct from TRAIN (green) and p-dep TEST (orange)

    header_fill = PatternFill("solid", fgColor=theme_color)
    header_font = Font(name=font_name, bold=True, color="FFFFFF", size=10)
    title_font = Font(name=font_name, bold=True, size=13, color=theme_color)
    subtitle_font = Font(name=font_name, italic=True, size=9, color="555555")
    cell_font = Font(name=font_name, size=10)
    border = Border(*(Side(style="thin", color="D9D9D9") for _ in range(4)))

    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = "Results"

    configs_text = "; ".join(f"{s}={r}" for s, r in BEST_CONFIGS.items())

    worksheet.cell(row=1, column=1,
                   value=f"Fixed Parameter-Free BEST_CONFIGS applied to 5-Fold "
                         f"held-out TEST (20%), seed={SEED}").font = title_font
    worksheet.cell(row=2, column=1,
                   value=f"{len(BEST_CONFIGS)} systems x {N_FOLDS} disjoint folds = "
                         f"{len(report)} rows. Each system uses exactly one fixed "
                         "rule, selected on TRAIN and never re-selected on TEST "
                         "(no p involved for this family). Normalization fit on "
                         "each fold's TRAIN submatrix, applied to that fold's TEST "
                         "submatrix.").font = subtitle_font
    worksheet.cell(row=3, column=1, value=f"Configurations used: {configs_text}").font = subtitle_font

    header_row = 5
    for col_idx, column in enumerate(report.columns, start=1):
        cell = worksheet.cell(row=header_row, column=col_idx, value=column)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    for row_offset, record in enumerate(report.itertuples(index=False), start=1):
        for col_idx, value in enumerate(record, start=1):
            column = report.columns[col_idx - 1]
            cell = worksheet.cell(row=header_row + row_offset, column=col_idx, value=value)
            cell.font = cell_font
            cell.border = border
            if column in ("EER (%)", "Rank-1 @ FAR=1%"):
                cell.number_format = "0.0000"

    worksheet.freeze_panes = worksheet.cell(row=header_row + 1, column=1)
    _autofit_columns(worksheet, len(report.columns), header_row, len(report))

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / OUTPUT_FILENAME_TEMPLATE.format(seed=SEED)
    workbook.save(output_path)
    return output_path


def _autofit_columns(worksheet, n_columns: int, header_row: int,
                     n_data_rows: int, minimum: int = 10, maximum: int = 30) -> None:
    for col_idx in range(1, n_columns + 1):
        letter = get_column_letter(col_idx)
        longest = max(
            (len(str(worksheet.cell(row=r, column=col_idx).value))
             for r in range(header_row, header_row + n_data_rows + 1)
             if worksheet.cell(row=r, column=col_idx).value is not None),
            default=minimum,
        )
        worksheet.column_dimensions[letter].width = min(max(longest + 2, minimum), maximum)


# ==============================================================================
#  ENTRY POINT -- no argparse, just run it.
# ==============================================================================

if __name__ == "__main__":
    print(f"Applying {len(BEST_CONFIGS)} fixed parameter-free rules to the TEST "
          f"folds (seed={SEED}, {N_FOLDS} folds) -- no sweep.")
    for system, rule in BEST_CONFIGS.items():
        print(f"  {system}: {rule}")

    report = evaluate_best_configs_on_test()

    output_path = write_report(report)
    print(f"\nReport written to: {output_path}")
    print(report.to_string(index=False))


Applying 8 fixed parameter-free rules to the TEST folds (seed=62, 5 folds) -- no sweep.
  S1: Rule3
  S2: Rule5
  S3: Rule5
  S4: Rule5
  S5: Rule5
  S6: Rule8
  S7: Rule5
  S8: Rule5
  Fold 1/5 done (104 test subjects).
  Fold 2/5 done (104 test subjects).
  Fold 3/5 done (103 test subjects).
  Fold 4/5 done (103 test subjects).
  Fold 5/5 done (103 test subjects).

Report written to: Results\ParamFree_BestConfigs_TEST_seed62.xlsx
 Fold System  Rule  EER (%)  Rank-1 @ FAR=1%  N_test
    1     S1 Rule3   1.9231          96.1538     104
    2     S1 Rule3   5.7692          87.5000     104
    3     S1 Rule3   4.9020          92.1569     103
    4     S1 Rule3   3.8835          88.3495     103
    5     S1 Rule3   1.9417          87.3786     103
    1     S2 Rule5   0.9615          99.0385     104
    2     S2 Rule5   0.9615          99.0385     104
    3     S2 Rule5   0.9804          99.0196     103
    4     S2 Rule5   0.9709          99.0291     103
    5     S2 Rule5   0.9709       